In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import imageio_ffmpeg    # <--- 如果没有，需要pip安装
plt.rcParams['animation.ffmpeg_path'] = imageio_ffmpeg.get_ffmpeg_exe()

In [ ]:
L = 400
C = 1
dx = 1.0
dt = 0.5
nt = 800  

x = np.arange(0, L, dx)
nx = len(x)

In [ ]:
def lax_wendroff(rho, C, dt, dx):
    rho_new = np.zeros_like(rho)
    r = dt / dx
    
    for j in range(1, nx-1):
        #  F = C * rho
        F_jp1 = C * rho[j+1]  
        F_jm1 = C * rho[j-1]  
        F_j = C * rho[j]      
        
       
        rho_jp_half = 0.5 * (rho[j+1] + rho[j])
        rho_jm_half = 0.5 * (rho[j] + rho[j-1])
        
       
        Fp_jp_half = C
        Fp_jm_half = C
        
       
        term1 = r * (F_jp1 - F_jm1) / 2
        term2 = (r**2) / 2 * (Fp_jp_half * (F_jp1 - F_j) - Fp_jm_half * (F_j - F_jm1))
        
        rho_new[j] = rho[j] - term1 + term2
    
    
    rho_new[0] = rho_new[1]
    rho_new[-1] = rho_new[-2]
    
    return rho_new

In [ ]:
def rho0_a(x, L):
    rho = np.zeros_like(x)
    mask = (x > L/4) & (x < L/2)
    rho[mask] = 4 * (x[mask] - L/4)**2 / L**2
    return rho


def rho0_b(x, L):
    rho = np.zeros_like(x)
    mask = (x > L/4) & (x < 3*L/4)
    rho[mask] = (L/4 - np.abs(x[mask] - L/2))*(4/L)
    return rho

In [ ]:
def simulate(rho0_func, C_val, case_name):
    rho_hist = []
    rho_curr = rho0_func(x, L)
    rho_hist.append(rho_curr.copy())
    
    for n in range(nt):
        rho_next = lax_wendroff(rho_curr, C_val, dt, dx)
        rho_curr = rho_next
        rho_hist.append(rho_curr.copy())
    
    return rho_hist


def create_animation(rho_hist, case_name, frame_step=5):
    fig, ax = plt.subplots(figsize=(10, 6))
    line, = ax.plot(x, rho_hist[0], 'b-', linewidth=2)
    ax.set_xlim(0, L)
    ax.set_ylim(0, max(np.max(rho) for rho in rho_hist) * 1.1)
    ax.set_xlabel('Position x')
    ax.set_ylabel('Density ρ')
    ax.set_title(f'Traffic Flow - {case_name}')
    ax.grid(True)
    
    frames = range(0, len(rho_hist), frame_step)
    
    def animate(i):
        line.set_ydata(rho_hist[i])
        return line,
    
    anim = FuncAnimation(fig, animate, frames=frames, 
                        interval=50, blit=True)
    
    plt.close(fig)
    return anim

In [ ]:
print("Simulating case (a)...")
rho_hist_a = simulate(rho0_a, C, "Case (a)")
anim_a = create_animation(rho_hist_a, "Case (a)", frame_step=5)
anim_a.save('output.mp4', writer='ffmpeg', fps=20)   # 保存MP4
HTML(anim_a.to_jshtml())

In [ ]:
print("Simulating case (b)...")
rho_hist_b = simulate(rho0_b, C, "Case (b)")
anim_b = create_animation(rho_hist_b, "Case (b)", frame_step=5)
HTML(anim_b.to_jshtml())

In [ ]:
print("Simulating case (c)...")
rho_hist_c = simulate(rho0_b, -C, "Case (c)")
anim_c = create_animation(rho_hist_c, "Case (c)", frame_step=5)
HTML(anim_c.to_jshtml())

# Von Neumann Stability Analysis



## General Solution

For the advection equation:

$$\frac{\partial u}{\partial t} = -c \frac{\partial u}{\partial x}$$

We use the von Neumann stability analysis method:

- **Deriving the Relation Among $u$ at different steps**
- **Von Neumann Stability Analysis**

Assume solution form:

$$u(n, r) = A^n e^{ikrh}$$

- **Obtain $A$ by plugging $u(n, r)$ into the relation above**

**Stability Criteria:**
- **Stable:** $|A(k)| \leq 1$, for all  $k$
- **Neutrally Stable:** $|A(k)| = 1$, for all  $k$
- **Unstable:** $ |A(k)| > 1$, for all  $k$

## (a) Forward Time Centered Space (FTCS)

**Discretization Scheme:**

$$\frac{u(n+1,r)-u(n,r)}{\tau} = -c \frac{u(n,r+1)-u(n,r-1)}{2h}$$

**Numerical Scheme:**

$$u(n+1,r) = u(n,r) - \frac{c\tau}{2h} [u(n,r+1)-u(n,r-1)]$$

**Substitute $u(n,r) = A^n e^{ikrh}$:**

$$A = 1 - \frac{c\tau}{2h} \left( e^{ikh} - e^{-ikh} \right) = 1 - i \frac{c\tau}{h} \sin(kh)$$

**Amplification Factor:**

$$|A| = \sqrt{1 + \left( \frac{c\tau}{2h} \right)^2 \sin^2(kh)} \geq 1$$

**Conclusion: Unstable**

## (b) Forward Time Forward Space (FTFS)

**Discretization Scheme:**

$$\frac{u(n+1,r)-u(n,r)}{\tau} = -c \frac{u(n,r+1)-u(n,r)}{h}$$

**Numerical Scheme:**

$$u(n+1,r) = u(n,r) - \frac{c\tau}{h} [u(n,r+1)-u(n,r)]$$

**Substitute $u(n,r) = A^n e^{ikrh}$:**

$$A = 1 - \frac{c\tau}{h} \left( e^{ikh} - 1 \right)$$

**Amplification Factor:**

$$|A|^2 = 1 + 2c \frac{\tau}{h} (1 - \cos kh)(1 + \frac{\tau}{h}) \quad \text{for } 0 \leq |c \frac{\tau}{h}| \leq 1$$

**Conclusion: Unstable** (since $|A|^2 > 1$ for some $k$)

## (c) Centered Time Centered Space (CTCS) - Leap-Frog Method

**Discretization Scheme:**

$$\frac{u(n+1,r)-u(n-1,r)}{2\tau} = -c \frac{u(n,r+1)-u(n,r-1)}{2h}$$

**Numerical Scheme:**

$$u(n+1,r) = u(n-1,r) - \frac{c\tau}{h} [u(n,r+1)-u(n,r-1)]$$

**Substitute $u(n,r) = A^n e^{ikrh}$:**

$$A - A^{-1} = -\frac{c\tau}{h} (e^{ikh} - e^{-ikh})$$

$$A^2 + 2 \frac{c\tau}{h} \sin(kh) - 1 = 0$$

**Characteristic Equation Solution:**

$$A = -i \frac{c\tau}{h} \sin(kh) \pm \sqrt{1 - \left(\frac{c\tau}{h}\right)^2 \sin^2(kh)}$$

**Stability Analysis:**

- **Case 1:** $\frac{c\tau}{h} \sin(kh) \leq 1$

  $$|A|^2 = \left(\frac{c\tau}{h}\right)^2 \sin^2(kh) + 1 - \left(\frac{c\tau}{h}\right)^2 \sin^2(kh) = 1$$

- **Case 2:** $\frac{c\tau}{h} \sin(kh) > 1$

  $$\exists |A| = \frac{c\tau}{h} \sin kh \pm \sqrt{\left(\frac{c\tau}{h} \sin kh\right)^2 - 1} > 1$$

**Stability Condition:** $|c\frac{\tau}{h}| \leq 1$

**Conclusion: Neutrally Stable** (when stability condition is satisfied)

## (d) Lax-Wendroff Method

**Numerical Scheme:**

$$u(n+1,r) = u(n,r) - \frac{c\tau}{2h} \left( u(n,r+1) - u(n,r-1) \right) + \frac{c^2 \tau^2}{2h^2} \left( u(n,r+1) + u(n,r-1) - 2u(n,r) \right)$$

**Substitute $u(n,r) = A^n e^{ikrh}$:**

$$A = 1 - i \frac{c\tau}{h} \sin(kh) - \left( \frac{c\tau}{h} \right)^2 [1 - \cos(kh)]$$

**Amplification Factor:**

$$|A|^2 = 1 - \left( \frac{c\tau}{h} \right)^2 \left( 1 - \left( \frac{c\tau}{h} \right)^2 \right) \left( 1 - \cos(kh) \right)^2$$

**Stability Analysis (CFL Condition):**

- $\frac{c\tau}{h} < 1$: $|A| \leq 1$ → **Stable**
- $\frac{c\tau}{h} = 1$: $|A| = 1$ → **Neutrally Stable**
- $\frac{c\tau}{h} > 1$: $ |A| > 1$ → **Unstable**

**Conclusion: Conditionally Stable** (stable when $\frac{c\tau}{h} \leq 1$)